<a href="https://colab.research.google.com/github/mvashi-sonic/AICapstoneProj/blob/dev/capstone_FAISS_retriever.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install sentence-transformers faiss-cpu

Mount The Drive

In [2]:
from google.colab import drive

# 1. Mount your Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


Import Required Packages

In [3]:
import json
import pickle
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

Sentence Tranformer

In [4]:
embedder = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
print(embedder.get_embedding_dimension())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

384


Read the Data

In [5]:
chunks = []

#with open("/content/drive/MyDrive/capstone_usable_qa_data/all_records.json", encoding="utf-8") as f:
with open("/content/drive/MyDrive/AI_capstone_training_data/all_records_final_formatted_3.json", encoding="utf-8") as f:
    records = json.load(f)
    for record in records["All_Records"]:
        chunks.append(record)


Create Pragraphs With Metadata

In [6]:
paragraphs = [
    chunk["reference"]
    for chunk in chunks
]
print(paragraphs[0][:300])
print(paragraphs[1][:300])
print(paragraphs[2][:300])

Section:7,Management's Discussion and Analysis of Financial Condition and Results of Operations (MD&A)
CompanyName:Adobe
Ticker:ADBE
Year:2025
&#8226; Customer journeys . Our products help businesses manage, test, target and personalize customer journeys delivered as campaigns across B2B and B2C use
Section:1A,Risk Factors
CompanyName:Vertex
Ticker:VRTX
Year:2025
. In February 2023, the U.S. Administration addressed access for cell and gene therapies in diseases such as SCD through the CMS program known as The Cell and Gene Therapy Access Model (&#8220;CGT Access Model&#8221;). The CGT Access M
Section:8,Financial Statements and Supplementary Data
CompanyName:Tesla
Ticker:TSLA
Year:2025
The above table does not include vehicle sales to customers or leasing partners with a resale value guarantee as the cash payments were received upfront. For our solar PPA arrangements, customers are charge


Create Embeddings

In [7]:
embeddings = embedder.encode(
    paragraphs,
    batch_size=128,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings = np.asarray(
    embeddings,
    dtype=np.float32
)
print(embeddings.shape)

Batches:   0%|          | 0/977 [00:00<?, ?it/s]

(124936, 384)


In [8]:

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
print("FAISS vectors:", index.ntotal)
print("Number of chunks:", len(chunks))
print("Number of embeddings:", len(embeddings))


FAISS vectors: 124936
Number of chunks: 124936
Number of embeddings: 124936


Write Indexes

In [9]:


faiss.write_index(
    index,
    "financial_reports.index"
)

In [10]:
#Check the file

import os

size_mb = os.path.getsize(
    "financial_reports.index"
) / (1024 * 1024)

print(f"Index size: {size_mb:.2f} MB")

Index size: 183.01 MB


In [11]:
metadata = []

for chunk in chunks:
    #print(f"{chunk["metadata"]}")
    tks = chunk["metadata"].split(" ")
    ticker = (tks[1]).split("-")[1]
    section = tks[len(tks)-1].split("-")[1]
    date = tks[len(tks)-2].split("-")[1]
    company = tks[2].split("-")[1]
    metadata.append({
        "ticker": ticker,
        "section": section,
        "year": date,
        "reference": chunk["reference"]
    })

In [12]:
with open("financial_reports_metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

In [8]:
import os

print(
    "File size:",
    os.path.getsize("financial_reports.index"),
    "bytes"
)

File size: 45 bytes


Read Indexes

In [13]:
index = faiss.read_index(
    "financial_reports.index"
)
with open("financial_reports_metadata.pkl","rb") as f:
    metadata = pickle.load(f)
index = faiss.IndexFlatIP(384)
index.add(embeddings)

Search For Top 50 Answers Based On Question Encoding

In [17]:
question = "What is Apple's revenue"

query_embedding = embedder.encode(
    [question],
    convert_to_numpy=True#,
    #normalize_embeddings=True
)

print(query_embedding.shape)
print(query_embedding[0][:10])

(1, 384)
[ 0.02150242  0.00418727  0.0103201  -0.04345614 -0.01762768 -0.0085749
  0.09277232  0.02231134  0.07692822  0.0481977 ]


In [18]:
scores, ids = index.search(
    query_embedding,
    k=50
)
print(scores)
print(ids)

[[0.6725098  0.6725098  0.6725098  0.6603533  0.6603533  0.6603533
  0.6603533  0.6603533  0.6603533  0.6603533  0.6603533  0.6603533
  0.6603533  0.6603533  0.6603533  0.6603533  0.6603533  0.6603533
  0.6603533  0.6603533  0.65736806 0.65736806 0.65736806 0.65736806
  0.65736806 0.65736806 0.65736806 0.65736806 0.65736806 0.65736806
  0.65736806 0.65736806 0.65736806 0.65736806 0.65736806 0.65736806
  0.65736806 0.65736806 0.65736806 0.65736806 0.65736806 0.65736806
  0.65736806 0.65736806 0.64648163 0.64648163 0.64648163 0.64648163
  0.64648163 0.64648163]]
[[123221 112761  17441 120586 115211 112280 105758  85464  81267  69723
   64760  56888  54775  52623  47444  44529  35157  30476  14520  14088
  115534  98124  96856  93804  93529  87513  87001  83207  72703  71842
   66187  65575  47951  47359  46272  43954  35012  29396  20398  19870
   16412  14960   9869   6519 108464 105415  89121  77270  73997  73338]]


In [19]:
for score, idx in zip(scores[0], ids[0]):
    chunk = metadata[idx]

    file_output = open("retrieved_results1.jsonl", "a")
    dict = {}
    dict["score"] = f"{score:.4f}"
    dict["ticker"] = chunk["ticker"]
    dict["section"] = chunk["section"]
    dict["year"] = chunk["section"]
    dict["reference"] = chunk["reference"]
    json.dump(dict, file_output)





Copy Results To Drive

In [16]:
#keep
#Copy data to drive
import os
import shutil
from google.colab import drive

# 2. Define source and destination folders
# Replace 'my_folder' with the exact folder path where your .json files currently are
source_dir = '.'
# Replace 'My Drive/TargetFolder' with the Drive folder you want to copy to
destination_dir = '/content/drive/MyDrive/AI_capstone_training_data/FAISS_Retriever'

# Create the destination directory if it doesn't exist
os.makedirs(destination_dir, exist_ok=True)

# 3. Find and copy all .json files
for filename in os.listdir(source_dir):
    if filename.endswith('retrieved_results.json'):
        source_file = os.path.join(source_dir, filename)
        destination_file = os.path.join(destination_dir, filename)

        shutil.copy2(source_file, destination_file)
        print(f"Copied: {filename}")

print("All .json files copied successfully!")

All .json files copied successfully!
